# 📓 Bitácora 03: La Prueba de Fuego

Un modelo que solo funciona en el set de entrenamiento no nos sirve. Aquí validamos el Agente G68 con frases de la vida real, incluyendo ironía extrema.

In [2]:
import sys
import os
import joblib
import numpy as np

# Configuración de rutas robusta
ROOT_PATH = os.path.abspath(os.path.join(os.getcwd(), '..'))
SRC_PATH = os.path.join(ROOT_PATH, 'src')
APP_PATH = os.path.join(SRC_PATH, 'app')
MODELS_PATH = os.path.join(ROOT_PATH, 'data', 'models')

if SRC_PATH not in sys.path: sys.path.append(SRC_PATH)
if APP_PATH not in sys.path: sys.path.append(APP_PATH)

from motor_hibrido import enriquecer_respuesta
from engine.sentiment_engine import SentimentEngine

# Inicialización corregida con ruta de modelos
engine = SentimentEngine(model_dir=MODELS_PATH)
print("✅ Motor G68 cargado y listo para inferencia.")

✅ Modelos ML cargados exitosamente desde: c:\ALURA - ONE\1. CIENCIA DE DATOS\HACKATHON\sentiment-api-G68\ml-python\data\models
✅ Motor G68 cargado y listo para inferencia.


### 1. Casos de Éxito y Retos Técnicos
Probamos el motor con una frase que normalmente engañaría a una IA tradicional.

In [3]:
frases_test = [
    "Excelente, el baño oliendo a cloaca y no hay agua caliente. Un aplauso.",
    "La cama es una nube, volvería mañana mismo.",
    "El hotel es normal, ni bueno ni malo."
]

for f in frases_test:
    pred_ia, prob_ia = engine.predict_raw(f)
    # Pasamos el engine para obtener todas las features y explicabilidad
    res = enriquecer_respuesta(f, pred_ia, prob_ia, engine)
    print(f"\n📝 Texto: {f}")
    print(f"🤖 G68: {res['prevision']} | Prob: {res['probabilidad']:.2f}")
    
    # Explicabilidad básica según contrato
    print(f"📌 Top Features: {res['top_features']}")


🔍 [G68 AUDIT] Texto: 'Excelente, el baño oliendo a cloaca y no hay agua caliente. ...'
   ├─ IA Sugiere: 1 (0.1199)
   ├─ Ajuste Semántico: -1.48
   ├─ Señales: ['agua caliente', 'no hay', 'oliendo a cloaca', 'excelente']
   ├─ Áreas: ['General', 'Operaciones']
   └─ VEREDICTO: Negativo (0.9900)

📝 Texto: Excelente, el baño oliendo a cloaca y no hay agua caliente. Un aplauso.
🤖 G68: [-] Negativo | Prob: 0.99
📌 Hallazgos: ['Contexto: excelente', 'Contexto: no hay', 'Contexto: oliendo a cloaca', 'Servicios/Agua (agua caliente)']

🔍 [G68 AUDIT] Texto: 'La cama es una nube, volvería mañana mismo....'
   ├─ IA Sugiere: 0 (0.1450)
   ├─ Ajuste Semántico: 0.50
   ├─ Señales: ['nube']
   ├─ Áreas: ['General']
   └─ VEREDICTO: Positivo (0.8500)

📝 Texto: La cama es una nube, volvería mañana mismo.
🤖 G68: [+] Positivo | Prob: 0.85
📌 Hallazgos: ['Contexto: nube']

🔍 [G68 AUDIT] Texto: 'El hotel es normal, ni bueno ni malo....'
   ├─ IA Sugiere: 0 (0.4015)
   ├─ Ajuste Semántico: -0.60
   ├─ Seña

### 2. Conclusión de las Pruebas
El motor híbrido logra neutralizar el sarcasmo gracias a su capa de reglas, manteniendo una precisión alta incluso en frases con conectores de contraste.